# Using LangChain Python Library with HuggingFace Hosted
Examples for non-streaming and streaming calls using Hugging Face models.


## Setup
This notebook shows examples for non-streaming and simulated streaming calls using Hugging Face's hosted Inference API in conversational mode. Requires `HF_TOKEN` environment variable.

In [ ]:
# imports
import os

from dotenv import load_dotenv
load_dotenv(override=True)

# LangChain imports (third-party)
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# local helpers
from streaming_utils import get_chunk_content


# depending on model, LangChain requires different instanciation
def get_llm_instance(model):
    # The endpoint requires a raw token string, so we pull it from the environment here.
    hf_llm = HuggingFaceEndpoint(
        model=model,
        task="conversational",
        huggingfacehub_api_token=os.getenv("HF_TOKEN"),
    )
    return ChatHuggingFace(llm=hf_llm)


def build_hf_prompt(messages):
    """Build a single plain-text prompt from a list of message objects (same as local notebook)."""
    parts = []
    for msg in messages:
        content = msg.content if hasattr(msg, "content") else str(msg)
        content = content.strip()
        if content:
            parts.append(content)
    return "\n\n".join(parts) + "\n\nAnswer:"

In [ ]:
# model
model_name = "meta-llama/Llama-3.1-8B-Instruct"

# prepare message
system_message = "You are a helpful assistant."
user_message = "Tell me a joke about programming."

# the composed messages is different for the LangChain
messages = [SystemMessage(content=system_message), HumanMessage(content=user_message)]

## Non-Streaming

In [ ]:
def message(messages, model, **kwargs):
    llm_instance = get_llm_instance(model)
    # Build a single text prompt to match the local notebook API
    prompt = build_hf_prompt(messages)

    used_kwargs = dict(kwargs)

    response = llm_instance.invoke(prompt, **used_kwargs)

    # Normalize response to a plain string for printing/consumption
    if hasattr(response, "content"):
        return response.content
    if hasattr(response, "message") and hasattr(response.message, "content"):
        return response.message.content
    return str(response)


# simple test call
print(f"Using model: {model_name}")
response = message(messages, model_name)
print(response)


## Simulated Streaming
Here we buffer chunks from the hosted endpoint rather than performing true token-by-token streaming. Use this when the provider returns chunked or buffered responses.

In [ ]:
def simulated_stream_message(messages, model_name, **kwargs):
    llm_instance = get_llm_instance(model_name)
    full_response = ""

    # Build a single text prompt to match the local notebook API
    prompt = build_hf_prompt(messages)

    used_kwargs = dict(kwargs)

    stream_iter = llm_instance.stream(prompt, **used_kwargs)

    for chunk in stream_iter:
        content = get_chunk_content(chunk)
        print(content, end="", flush=True)
        full_response += content
    return full_response


# simple test call
print(f"Using model: {model_name}")
response = simulated_stream_message(messages, model_name)

## True Streaming
For true token-by-token streaming you need raw model access (e.g., local Transformers `TextIteratorStreamer`). Hosted conversational endpoints frequently buffer or provide chunked streams; true low-latency token streaming is generally only available when running the model locally or via a provider that explicitly exposes token streams (provider-specific).